# PHASE 2 KOMPONEN 2 — LANGKAH 3: SOCIAL PENETRATION METRICS
## Cross-Community Penetration & Cognitive Fusion

**Tujuan:** Mengukur seberapa jauh setiap narasi/topik menembus batas-batas komunitas.
Ini yang membedakan jurnal dari proceeding — bukan sekadar "jarak maksimal",
tapi "cognitive fusion" lintas komunitas.

**Input:**
- `Data_bert_with_topics.csv` — tweet dengan topic labels
- `network_nodes_with_community.csv` — node + community assignment dari Langkah 2
- `network_edges.csv` — weighted edge list dari Langkah 1

**Output:**
- `penetration_by_narrative.csv` — metrik penetrasi per narrative cluster
- `penetration_by_topic.csv` — metrik penetrasi per topic

**Metrik yang dihitung:**
1. Community Reach — berapa komunitas unik yang dijangkau
2. Cross-Community Edge Ratio — proporsi edge yang menembus batas komunitas
3. Community Hop Distance — seberapa "jauh" narasi menyebar antar komunitas
4. Perbandingan penetrasi: Affan Kurniawan vs Demo & DPR vs lainnya

## Cell 1: Install & Import

In [15]:
!pip install networkx python-louvain openpyxl -q

import pandas as pd
import networkx as nx
import numpy as np
from collections import Counter, defaultdict
from itertools import combinations
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully.")

All libraries imported successfully.


## Cell 2: Load All Data & Rebuild Graph

In [16]:
# Load tweet data dengan topic labels
df = pd.read_csv('Data_bert_with_topics.csv')

# Load node + community assignment
nodes_comm = pd.read_csv('network_nodes_with_community.csv')

# Load edge list
edges_df = pd.read_csv('network_edges.csv')

print(f"Tweets: {len(df)}")
print(f"Nodes with community: {len(nodes_comm)}")
print(f"Edges: {len(edges_df)}")

# Buat mapping username -> community
comm_map = dict(zip(nodes_comm['username'], nodes_comm['community']))

# Buat mapping username -> account_type
type_map = dict(zip(nodes_comm['username'], nodes_comm['account_type']))

# Rebuild directed graph
G = nx.DiGraph()
for _, row in edges_df.iterrows():
    G.add_edge(
        row['source'], row['target'],
        weight=row['weight'],
        dominant_type=row['dominant_type'],
        dominant_topic=row['dominant_topic']
    )

# Assign community & account_type ke nodes
for node in G.nodes():
    G.nodes[node]['community'] = comm_map.get(node, -1)
    G.nodes[node]['account_type'] = type_map.get(node, 'unknown')

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Unique communities: {len(set(comm_map.values()))}")

Tweets: 29289
Nodes with community: 11426
Edges: 13994
Graph: 11426 nodes, 13994 edges
Unique communities: 1667


## Cell 3: Narrative Mapping

Mapping dari 19 BERTopic topics ke 6 narrative clusters.
Mapping ini berdasarkan hasil Analysis 2.1.

In [17]:
# Narrative cluster mapping dari BERTopic results
narrative_map = {
    0: 'Demo & DPR',           # demo_dpr_gedung_mahasiswa
    1: 'Gerakan/Hashtag',      # indonesiagelap_resetindonesia
    2: 'Politik & Tuntutan',   # prabowo_dpr_presiden_tuntutan
    3: 'Kekerasan Aparat',    # polisi_polisipembunuh
    4: 'Ekonomi Rakyat',      # rakyat_tuntutan_pajak_gaji
    5: 'Affan Kurniawan',     # affan_kurniawan_almarhum
    6: 'Kekerasan Aparat',    # cops_1312_bastard
    7: 'Kekerasan Aparat',    # 1312_pembunuh_forever
    8: 'Gerakan/Hashtag',     # bubarkandprsontoloyo
    9: 'Keamanan & Respons',  # stay_safe_jaga
    10: 'Demo & DPR',         # temen_demo
    11: 'Kekerasan Aparat',   # anjing_polisi_1312
    12: 'Demo & DPR',         # demo_1312_dpr_budaya
    13: 'Kekerasan Aparat',   # 1312_medis_mata_mobil
    14: 'Gerakan/Hashtag',    # adilijokowi_indonesiagelap
    15: 'Gerakan/Hashtag',    # stopbayarpajak_bubarkandpr
    16: 'Ekonomi Rakyat',     # 80_indonesia_umkm
    17: 'Keamanan & Respons', # bendera_takut_pasang
    18: 'Keamanan & Respons'  # damai_indonesiaku_provokasi
}

# Apply ke tweet dataframe
df['narrative'] = df['topic'].map(narrative_map)

# Clean username untuk matching
df['username_clean'] = df['username'].str.strip().str.lstrip('@').str.lower()

print("Narrative distribution:")
print(df['narrative'].value_counts())
print(f"\nUnmapped (NaN): {df['narrative'].isna().sum()}")

Narrative distribution:
narrative
Demo & DPR            6909
Kekerasan Aparat      4809
Gerakan/Hashtag       3891
Politik & Tuntutan    2506
Ekonomi Rakyat        2151
Affan Kurniawan       1595
Keamanan & Respons     677
Name: count, dtype: int64

Unmapped (NaN): 6751


## Cell 4: Assign Community to Each Tweet

Setiap tweet mendapat community assignment berdasarkan username penulisnya.

In [18]:
# Map community ke setiap tweet
df['community'] = df['username_clean'].map(comm_map)

# Cek coverage
matched = df['community'].notna().sum()
total = len(df)
print(f"Tweets with community assignment: {matched} ({matched/total*100:.1f}%)")
print(f"Tweets without community (user not in graph): {total - matched}")

# Filter hanya tweet yang punya community
df_comm = df[df['community'].notna()].copy()
df_comm['community'] = df_comm['community'].astype(int)
print(f"\nWorking dataset: {len(df_comm)} tweets across {df_comm['community'].nunique()} communities")

Tweets with community assignment: 14248 (48.6%)
Tweets without community (user not in graph): 15041

Working dataset: 14248 tweets across 1667 communities


## Cell 5: Metrik 1 — Community Reach per Narrative

Berapa banyak komunitas unik yang dijangkau oleh setiap narasi?
Ini mengukur "breadth" dari penetrasi.

In [19]:
# Community reach per narrative
reach_by_narrative = df_comm.groupby('narrative').agg(
    total_tweets=('username_clean', 'size'),
    unique_users=('username_clean', 'nunique'),
    communities_reached=('community', 'nunique')
).reset_index()

# Total communities in network untuk persentase
total_communities = df_comm['community'].nunique()
reach_by_narrative['reach_pct'] = (reach_by_narrative['communities_reached'] / total_communities * 100).round(1)

# Reach per user (efisiensi penetrasi)
reach_by_narrative['communities_per_user'] = (
    reach_by_narrative['communities_reached'] / reach_by_narrative['unique_users']
).round(3)

reach_by_narrative = reach_by_narrative.sort_values('communities_reached', ascending=False)

print("=" * 75)
print("COMMUNITY REACH PER NARRATIVE")
print("=" * 75)
print(f"Total communities in network: {total_communities}")
print()
print(f"{'Narrative':<22} {'Tweets':<8} {'Users':<8} {'Comms':<8} {'Reach%':<8} {'Comm/User':<10}")
print("-" * 64)
for _, row in reach_by_narrative.iterrows():
    print(f"{row['narrative']:<22} {row['total_tweets']:<8} {row['unique_users']:<8} "
          f"{row['communities_reached']:<8} {row['reach_pct']:<8} {row['communities_per_user']:<10}")

COMMUNITY REACH PER NARRATIVE
Total communities in network: 1667

Narrative              Tweets   Users    Comms    Reach%   Comm/User 
----------------------------------------------------------------
Demo & DPR             2482     1998     683      41.0     0.342     
Kekerasan Aparat       2276     1674     540      32.4     0.323     
Gerakan/Hashtag        2356     1064     253      15.2     0.238     
Ekonomi Rakyat         1277     995      154      9.2      0.155     
Politik & Tuntutan     1437     1059     122      7.3      0.115     
Affan Kurniawan        743      479      102      6.1      0.213     
Keamanan & Respons     206      158      68       4.1      0.43      


## Cell 6: Metrik 1b — Community Reach per Individual Topic

Sama seperti di atas tapi per topik individual (bukan narrative cluster).
Ini untuk granularitas lebih tinggi.

In [20]:
reach_by_topic = df_comm.groupby(['topic', 'topic_label']).agg(
    total_tweets=('username_clean', 'size'),
    unique_users=('username_clean', 'nunique'),
    communities_reached=('community', 'nunique')
).reset_index()

reach_by_topic['reach_pct'] = (reach_by_topic['communities_reached'] / total_communities * 100).round(1)
reach_by_topic['communities_per_user'] = (
    reach_by_topic['communities_reached'] / reach_by_topic['unique_users']
).round(3)

reach_by_topic = reach_by_topic.sort_values('communities_reached', ascending=False)

print("=" * 85)
print("COMMUNITY REACH PER TOPIC")
print("=" * 85)
print(f"{'Topic':<6} {'Label':<35} {'Tweets':<8} {'Comms':<8} {'Reach%':<8} {'C/User':<8}")
print("-" * 73)
for _, row in reach_by_topic.iterrows():
    label = str(row['topic_label'])[:33]
    print(f"{row['topic']:<6} {label:<35} {row['total_tweets']:<8} "
          f"{row['communities_reached']:<8} {row['reach_pct']:<8} {row['communities_per_user']:<8}")

COMMUNITY REACH PER TOPIC
Topic  Label                               Tweets   Comms    Reach%   C/User  
-------------------------------------------------------------------------
0      -1 | demo | rakyat | dpr            2218     577      34.6     0.326   
-1     Outlier                             3471     510      30.6     0.208   
6      5 | affan | kurniawan | almarhum    714      298      17.9     0.497   
1      0 | demo | dpr | gedung             1631     194      11.6     0.254   
3      2 | prabowo | dpr | presiden        916      159      9.5      0.271   
7      6 | cops | 1312 | bastard           363      156      9.4      0.456   
4      3 | polisi | polisipembunuh | pol   1177     153      9.2      0.154   
2      1 | indonesiagelap | resetindones   1437     122      7.3      0.115   
5      4 | rakyat | tuntutan | pajak       743      102      6.1      0.213   
12     11 | anjing | polisi | 1312         152      98       5.9      0.667   
10     9 | stay | safe | jaga  

## Cell 7: Metrik 2 — Cross-Community Edge Ratio per Narrative

Untuk setiap narasi, berapa proporsi edge yang menghubungkan node di komunitas berbeda?
Rasio tinggi = narasi yang berhasil menembus echo chamber.

In [21]:
# Rebuild raw edges dengan topic info untuk analisis per narrative
# Kita perlu edges_df yang punya dominant_topic, lalu map ke narrative

edges_with_comm = edges_df.copy()
edges_with_comm['source_comm'] = edges_with_comm['source'].map(comm_map)
edges_with_comm['target_comm'] = edges_with_comm['target'].map(comm_map)
edges_with_comm['narrative'] = edges_with_comm['dominant_topic'].map(narrative_map)

# Filter edges yang punya community di kedua sisi
edges_valid = edges_with_comm.dropna(subset=['source_comm', 'target_comm', 'narrative']).copy()
edges_valid['is_cross_community'] = edges_valid['source_comm'] != edges_valid['target_comm']

print(f"Total valid edges: {len(edges_valid)}")
print(f"Cross-community edges: {edges_valid['is_cross_community'].sum()}")
print()

# Cross-community ratio per narrative
cross_ratio = edges_valid.groupby('narrative').agg(
    total_edges=('is_cross_community', 'size'),
    cross_edges=('is_cross_community', 'sum'),
    total_weight=('weight', 'sum')
).reset_index()

cross_ratio['cross_ratio'] = (cross_ratio['cross_edges'] / cross_ratio['total_edges'] * 100).round(1)
cross_ratio = cross_ratio.sort_values('cross_ratio', ascending=False)

print("=" * 65)
print("CROSS-COMMUNITY EDGE RATIO PER NARRATIVE")
print("=" * 65)
print(f"{'Narrative':<22} {'Total Edges':<13} {'Cross Edges':<13} {'Cross%':<8} {'Weight':<8}")
print("-" * 64)
for _, row in cross_ratio.iterrows():
    print(f"{row['narrative']:<22} {int(row['total_edges']):<13} {int(row['cross_edges']):<13} "
          f"{row['cross_ratio']:<8} {int(row['total_weight']):<8}")

Total valid edges: 10342
Cross-community edges: 1593

CROSS-COMMUNITY EDGE RATIO PER NARRATIVE
Narrative              Total Edges   Cross Edges   Cross%   Weight  
----------------------------------------------------------------
Politik & Tuntutan     1602          396           24.7     2403    
Gerakan/Hashtag        2916          597           20.5     4888    
Keamanan & Respons     141           26            18.4     202     
Ekonomi Rakyat         1072          155           14.5     1618    
Affan Kurniawan        422           58            13.7     578     
Kekerasan Aparat       2055          225           10.9     3095    
Demo & DPR             2134          136           6.4      3120    


## Cell 8: Metrik 3 — Community Hop Distance per Narrative

Mengukur "jarak" antar komunitas yang terhubung oleh narasi tertentu.
Hop distance dihitung pada community-level graph:
- Node = community
- Edge = ada interaksi antar member dua community
- Shortest path = minimum hops untuk menghubungkan komunitas terjauh

Ini adalah rekonceptualisasi "maximal distance" dari proceeding.

In [22]:
# Bangun community-level graph per narrative
print("=" * 65)
print("COMMUNITY HOP DISTANCE PER NARRATIVE")
print("=" * 65)
print()

hop_results = []

for narrative in sorted(df_comm['narrative'].dropna().unique()):
    # Filter edges untuk narrative ini
    narr_edges = edges_valid[edges_valid['narrative'] == narrative]

    # Filter hanya cross-community edges
    cross_edges = narr_edges[narr_edges['is_cross_community']]

    if len(cross_edges) == 0:
        hop_results.append({
            'narrative': narrative,
            'community_nodes': 0,
            'community_edges': 0,
            'max_hop': 0,
            'avg_hop': 0,
            'diameter': 0
        })
        continue

    # Build community-level graph
    CG = nx.Graph()
    for _, row in cross_edges.iterrows():
        c1 = int(row['source_comm'])
        c2 = int(row['target_comm'])
        if CG.has_edge(c1, c2):
            CG[c1][c2]['weight'] += row['weight']
        else:
            CG.add_edge(c1, c2, weight=row['weight'])

    # Juga tambahkan komunitas yang punya tweet di narrative ini tapi mungkin tidak ada cross-edge
    narr_communities = set(df_comm[df_comm['narrative'] == narrative]['community'].unique())
    for c in narr_communities:
        CG.add_node(c)

    # Hitung hop distance pada largest connected component
    if CG.number_of_nodes() > 1:
        largest_cc = max(nx.connected_components(CG), key=len)
        CG_sub = CG.subgraph(largest_cc).copy()

        if CG_sub.number_of_nodes() > 1:
            # Diameter = maximum shortest path
            try:
                diameter = nx.diameter(CG_sub)
            except:
                diameter = 0

            # Average shortest path
            try:
                avg_path = nx.average_shortest_path_length(CG_sub)
            except:
                avg_path = 0
        else:
            diameter = 0
            avg_path = 0
    else:
        diameter = 0
        avg_path = 0
        largest_cc = set()

    hop_results.append({
        'narrative': narrative,
        'community_nodes': CG.number_of_nodes(),
        'community_edges': CG.number_of_edges(),
        'largest_cc_size': len(largest_cc) if largest_cc else 0,
        'max_hop': diameter,
        'avg_hop': round(avg_path, 2)
    })

hop_df = pd.DataFrame(hop_results).sort_values('max_hop', ascending=False)

print(f"{'Narrative':<22} {'Comm Nodes':<12} {'Comm Edges':<12} {'LCC Size':<10} {'Max Hop':<9} {'Avg Hop':<8}")
print("-" * 73)
for _, row in hop_df.iterrows():
    print(f"{row['narrative']:<22} {row['community_nodes']:<12} {row['community_edges']:<12} "
          f"{row.get('largest_cc_size', 0):<10} {row['max_hop']:<9} {row['avg_hop']:<8}")

COMMUNITY HOP DISTANCE PER NARRATIVE

Narrative              Comm Nodes   Comm Edges   LCC Size   Max Hop   Avg Hop 
-------------------------------------------------------------------------
Affan Kurniawan        102          37           25         7         2.84    
Demo & DPR             683          83           33         5         2.27    
Keamanan & Respons     70           23           20         5         2.63    
Ekonomi Rakyat         154          87           30         4         2.0     
Kekerasan Aparat       541          118          32         4         1.93    
Gerakan/Hashtag        255          166          37         3         1.83    
Politik & Tuntutan     123          98           34         3         1.87    


## Cell 9: Perbandingan Head-to-Head — Affan Kurniawan vs Demo & DPR

Ini yang menjawab pertanyaan riset: apakah narasi kekerasan aparat (Affan Kurniawan)
menembus lebih banyak komunitas dibanding narasi ekonomi (Demo & DPR)?

In [23]:
print("=" * 65)
print("HEAD-TO-HEAD: AFFAN KURNIAWAN vs DEMO & DPR")
print("=" * 65)
print()

# Ambil data kedua narasi
for narr in ['Affan Kurniawan', 'Demo & DPR', 'Kekerasan Aparat', 'Ekonomi Rakyat']:
    print(f"--- {narr} ---")

    # Community reach
    reach_row = reach_by_narrative[reach_by_narrative['narrative'] == narr]
    if len(reach_row) > 0:
        r = reach_row.iloc[0]
        print(f"  Tweets: {r['total_tweets']}, Users: {r['unique_users']}")
        print(f"  Communities reached: {r['communities_reached']} ({r['reach_pct']}%)")
        print(f"  Communities per user: {r['communities_per_user']}")

    # Cross-community ratio
    cross_row = cross_ratio[cross_ratio['narrative'] == narr]
    if len(cross_row) > 0:
        c = cross_row.iloc[0]
        print(f"  Cross-community edge ratio: {c['cross_ratio']}%")

    # Hop distance
    hop_row = hop_df[hop_df['narrative'] == narr]
    if len(hop_row) > 0:
        h = hop_row.iloc[0]
        print(f"  Max community hop: {h['max_hop']}")
        print(f"  Avg community hop: {h['avg_hop']}")
    print()

# Normalized comparison (penetrasi per 100 users)
print("=" * 65)
print("NORMALIZED PENETRATION (per 100 users)")
print("=" * 65)
print()

for narr in ['Affan Kurniawan', 'Demo & DPR', 'Kekerasan Aparat', 'Ekonomi Rakyat']:
    reach_row = reach_by_narrative[reach_by_narrative['narrative'] == narr]
    if len(reach_row) > 0:
        r = reach_row.iloc[0]
        comms_per_100 = (r['communities_reached'] / r['unique_users']) * 100
        print(f"{narr:<22}: {comms_per_100:.1f} communities per 100 users")

HEAD-TO-HEAD: AFFAN KURNIAWAN vs DEMO & DPR

--- Affan Kurniawan ---
  Tweets: 743, Users: 479
  Communities reached: 102 (6.1%)
  Communities per user: 0.213
  Cross-community edge ratio: 13.7%
  Max community hop: 7
  Avg community hop: 2.84

--- Demo & DPR ---
  Tweets: 2482, Users: 1998
  Communities reached: 683 (41.0%)
  Communities per user: 0.342
  Cross-community edge ratio: 6.4%
  Max community hop: 5
  Avg community hop: 2.27

--- Kekerasan Aparat ---
  Tweets: 2276, Users: 1674
  Communities reached: 540 (32.4%)
  Communities per user: 0.323
  Cross-community edge ratio: 10.9%
  Max community hop: 4
  Avg community hop: 1.93

--- Ekonomi Rakyat ---
  Tweets: 1277, Users: 995
  Communities reached: 154 (9.2%)
  Communities per user: 0.155
  Cross-community edge ratio: 14.5%
  Max community hop: 4
  Avg community hop: 2.0

NORMALIZED PENETRATION (per 100 users)

Affan Kurniawan       : 21.3 communities per 100 users
Demo & DPR            : 34.2 communities per 100 users
Keker

## Cell 10: Statistical Test — Community Reach Comparison

Apakah perbedaan penetrasi antar narasi signifikan secara statistik?
Kita gunakan chi-square test dan permutation test.

In [24]:
print("=" * 65)
print("STATISTICAL TESTS")
print("=" * 65)
print()

# Test 1: Per-user community diversity
# Untuk setiap user, hitung berapa komunitas berbeda yang mereka berinteraksi dengan
# (bukan cuma komunitas mereka sendiri, tapi komunitas target interaksi mereka)

# Assign narrative ke setiap user berdasarkan tweet dominan mereka
user_narrative = df_comm.groupby('username_clean')['narrative'].agg(
    lambda x: x.value_counts().index[0] if len(x.value_counts()) > 0 else 'unknown'
).to_dict()

# Untuk setiap user, hitung berapa komunitas berbeda yang mereka interact dengan
user_cross_comm = defaultdict(set)
for _, row in edges_valid.iterrows():
    source = row['source']
    target_comm = row['target_comm']
    source_comm = row['source_comm']
    if source_comm != target_comm:
        user_cross_comm[source].add(int(target_comm))

# Buat dataframe per user
user_data = []
for user in df_comm['username_clean'].unique():
    narr = user_narrative.get(user, 'unknown')
    cross_comms = len(user_cross_comm.get(user, set()))
    user_data.append({
        'username': user,
        'narrative': narr,
        'cross_community_reach': cross_comms
    })

user_df = pd.DataFrame(user_data)

# Kruskal-Wallis test: apakah cross-community reach berbeda antar narrative groups?
groups = []
group_labels = []
for narr in ['Affan Kurniawan', 'Demo & DPR', 'Kekerasan Aparat', 'Ekonomi Rakyat']:
    vals = user_df[user_df['narrative'] == narr]['cross_community_reach'].values
    if len(vals) > 0:
        groups.append(vals)
        group_labels.append(narr)
        print(f"{narr:<22}: n={len(vals)}, mean cross-comm reach={np.mean(vals):.3f}, median={np.median(vals):.0f}")

print()

# Kruskal-Wallis
if len(groups) >= 2:
    h_stat, p_val = stats.kruskal(*groups)
    print(f"Kruskal-Wallis H = {h_stat:.3f}, p = {p_val:.6f}")
    if p_val < 0.05:
        print("→ Significant difference in cross-community reach between narratives.")
    else:
        print("→ No significant difference.")

# Pairwise Mann-Whitney: Affan vs Demo & DPR
print()
affan_vals = user_df[user_df['narrative'] == 'Affan Kurniawan']['cross_community_reach'].values
dpr_vals = user_df[user_df['narrative'] == 'Demo & DPR']['cross_community_reach'].values

if len(affan_vals) > 0 and len(dpr_vals) > 0:
    u_stat, p_val = stats.mannwhitneyu(affan_vals, dpr_vals, alternative='two-sided')
    # Effect size r = Z / sqrt(N)
    n_total = len(affan_vals) + len(dpr_vals)
    z_score = stats.norm.ppf(1 - p_val/2)
    r_effect = z_score / np.sqrt(n_total)
    print(f"Mann-Whitney U (Affan vs Demo&DPR): U={u_stat:.0f}, p={p_val:.6f}, r={r_effect:.3f}")
    if p_val < 0.05:
        print("→ Significant difference.")
    else:
        print("→ No significant difference.")

STATISTICAL TESTS

Affan Kurniawan       : n=366, mean cross-comm reach=0.101, median=0
Demo & DPR            : n=1754, mean cross-comm reach=0.087, median=0
Kekerasan Aparat      : n=1462, mean cross-comm reach=0.144, median=0
Ekonomi Rakyat        : n=744, mean cross-comm reach=0.156, median=0

Kruskal-Wallis H = 18.264, p = 0.000388
→ Significant difference in cross-community reach between narratives.

Mann-Whitney U (Affan vs Demo&DPR): U=330222, p=0.047332, r=0.043
→ Significant difference.


## Cell 11: Account Type Contribution to Cross-Community Penetration

Siapa yang mendorong penetrasi lintas komunitas — grassroot, media, atau elite?
Ini penting untuk RQ2 (grassroots sebagai diffusion drivers).

In [25]:
# Untuk setiap cross-community edge, siapa source-nya?
cross_only = edges_valid[edges_valid['is_cross_community']].copy()
cross_only['source_type'] = cross_only['source'].map(type_map)

print("=" * 65)
print("WHO DRIVES CROSS-COMMUNITY PENETRATION?")
print("=" * 65)
print()

# Overall
print("Overall cross-community edge sources:")
src_counts = cross_only['source_type'].value_counts()
for t, c in src_counts.items():
    print(f"  {t}: {c} ({c/len(cross_only)*100:.1f}%)")

# Per narrative
print()
print("Cross-community edge sources per narrative:")
print(f"{'Narrative':<22} {'Grassroot%':<12} {'Media%':<12} {'Elite%':<12} {'Total':<8}")
print("-" * 66)

for narr in ['Affan Kurniawan', 'Demo & DPR', 'Kekerasan Aparat', 'Ekonomi Rakyat',
             'Gerakan/Hashtag', 'Politik & Tuntutan', 'Keamanan & Respons']:
    subset = cross_only[cross_only['narrative'] == narr]
    if len(subset) == 0:
        continue
    types = subset['source_type'].value_counts()
    gr = types.get('grassroot', 0)
    mn = types.get('media/news', 0)
    ep = types.get('elite/political', 0)
    total = len(subset)
    print(f"{narr:<22} {gr/total*100:>5.1f}%{'':>5} {mn/total*100:>5.1f}%{'':>5} {ep/total*100:>5.1f}%{'':>5} {total}")

WHO DRIVES CROSS-COMMUNITY PENETRATION?

Overall cross-community edge sources:
  grassroot: 1580 (99.2%)
  elite/political: 13 (0.8%)

Cross-community edge sources per narrative:
Narrative              Grassroot%   Media%       Elite%       Total   
------------------------------------------------------------------
Affan Kurniawan         91.4%        0.0%        8.6%      58
Demo & DPR             100.0%        0.0%        0.0%      136
Kekerasan Aparat        99.6%        0.0%        0.4%      225
Ekonomi Rakyat          97.4%        0.0%        2.6%      155
Gerakan/Hashtag        100.0%        0.0%        0.0%      597
Politik & Tuntutan      99.2%        0.0%        0.8%      396
Keamanan & Respons     100.0%        0.0%        0.0%      26


## Cell 12: Save All Outputs

In [26]:
# 1. Penetration by narrative
penetration_narr = reach_by_narrative.merge(
    cross_ratio[['narrative', 'cross_ratio', 'cross_edges', 'total_edges']],
    on='narrative', how='left'
).merge(
    hop_df[['narrative', 'max_hop', 'avg_hop', 'community_nodes', 'community_edges']],
    on='narrative', how='left'
)
penetration_narr.to_csv('penetration_by_narrative.csv', index=False)

# 2. Penetration by topic
reach_by_topic.to_csv('penetration_by_topic.csv', index=False)

# 3. Updated tweet data dengan community assignment
df_comm.to_csv('Data_with_community.csv', index=False)

print(f"Saved: penetration_by_narrative.csv ({len(penetration_narr)} narratives)")
print(f"Saved: penetration_by_topic.csv ({len(reach_by_topic)} topics)")
print(f"Saved: Data_with_community.csv ({len(df_comm)} tweets)")
print(f"\n--- Langkah 3 selesai. Phase 2 Komponen 2 COMPLETE. ---")
print(f"--- Ready untuk Phase 3: Multi-Dimensional Network Analysis. ---")

Saved: penetration_by_narrative.csv (7 narratives)
Saved: penetration_by_topic.csv (20 topics)
Saved: Data_with_community.csv (14248 tweets)

--- Langkah 3 selesai. Phase 2 Komponen 2 COMPLETE. ---
--- Ready untuk Phase 3: Multi-Dimensional Network Analysis. ---
